In [27]:
from io import StringIO
from io import BytesIO
import zipfile

import kml2geojson
import requests
import pandas as pd

In [28]:
url = "https://www.google.com/maps/d/kml?mid=1vP3B4hda5fYtW4GSoQVLJXJxqIM"

In [29]:
data = requests.get(url)
data = BytesIO(data.content)
zf = zipfile.ZipFile(data, "r")

In [30]:
# Usually KMZ contains doc.kml
kml_name = next((n for n in zf.namelist() if n.lower().endswith(".kml")), None)
if not kml_name:
    raise FileNotFoundError("No KML file found inside KMZ")

kml_bytes = zf.read(kml_name)   # raw bytes (still in memory)
kml_text = kml_bytes.decode("utf-8", errors="replace")

In [31]:
def _syntheticIds(length):
    return [f'LV000{i}' for i in range(1, length + 1)]



In [32]:
jsondata = kml2geojson.convert(StringIO(kml_text))

In [33]:
data = pd.json_normalize(jsondata[0]['features'])
data = data[[
    'properties.name',
    'geometry.coordinates'
]]
data['id'] = _syntheticIds(len(data))
data['lon'] = data['geometry.coordinates'].apply(lambda x: x[0])
data['lat'] = data['geometry.coordinates'].apply(lambda x: x[1])
data.drop("geometry.coordinates", axis=1, inplace=True)
data.rename(mapper={
    "properties.name": "name"
}, axis=1, inplace=True)
data = data [[
    "id", "name", "lat", "lon"
]]
data

,id,name,lat,lon
0,LV0001,Bābelīte,56.992043,24.221142
1,LV0002,Būšnieku ezers,57.438007,21.655018
2,LV0003,Salacgrīva,57.750240,24.345961
3,LV0004,Ainaži,57.858727,24.345961
4,LV0005,Lielais stropu ezers,55.903240,26.589554
5,LV0006,Zirga ezers,55.910351,27.180433
6,LV0007,Gaurata ezers,56.663727,23.293333
7,LV0008,Cieceres ezers,56.678253,22.562485
8,LV0009,Dienvidrietumu pludmale,56.491642,20.994251
9,LV00010,Saldus ezers,56.672096,22.506952
